# Different Ways to Handle NULL Values in Spark, PySpark, Hive & SQL (Real-Time Projects)

NULL values are common in real-world datasets due to missing information, system failures, optional fields, data integration issues, or delayed data arrival. Proper handling of NULL values is essential for maintaining data quality and ensuring accurate analytics.

This guide covers the most common techniques for handling NULL values across **Spark, PySpark, Hive, and SQL**, along with real-time project scenarios.

---

# 1. Remove Rows Containing NULL Values

## Use Case
Remove records with missing mandatory fields.

### PySpark

```python
df = df.na.drop()
```

Remove rows where specific columns contain NULLs.

```python
df = df.na.drop(subset=["customer_id", "email"])
```

### Hive / SQL

```sql
SELECT *
FROM customer
WHERE customer_id IS NOT NULL;
```

### Real-Time Example

Reject customer records without Customer ID.

---

# 2. Replace NULL Values with Default Values

## Use Case
Replace NULLs with a fixed value.

### PySpark

```python
df = df.na.fill(0)
```

Replace string columns

```python
df = df.na.fill("Unknown")
```

Replace multiple columns

```python
df = df.na.fill({
    "salary": 0,
    "city": "Unknown"
})
```

### SQL / Hive

```sql
SELECT
COALESCE(city,'Unknown') AS city
FROM employee;
```

### Real-Time Example

Replace missing city names with "Unknown".

---

# 3. Using `COALESCE()`

Returns the first non-null value.

### Spark SQL / Hive / SQL

```sql
SELECT
COALESCE(phone,mobile,'NA')
FROM customer;
```

### PySpark

```python
from pyspark.sql.functions import coalesce

df = df.withColumn(
    "contact",
    coalesce("phone","mobile")
)
```

### Real-Time Example

Customer may have either Phone or Mobile.

---

# 4. Using `IFNULL()` / `NVL()`

Replace NULL with another value.

### Hive

```sql
SELECT
NVL(salary,0)
FROM employee;
```

### MySQL

```sql
SELECT
IFNULL(salary,0)
FROM employee;
```

### Oracle

```sql
SELECT
NVL(salary,0)
FROM employee;
```

### Real-Time Example

Replace missing salaries with 0.

---

# 5. Using `CASE WHEN`

Complex NULL handling.

### SQL

```sql
SELECT

CASE

WHEN salary IS NULL
THEN 0

ELSE salary

END

FROM employee;
```

### Spark

```python
from pyspark.sql.functions import when,col

df = df.withColumn(
"salary",
when(col("salary").isNull(),0)
.otherwise(col("salary"))
)
```

### Real-Time Example

Different default values based on business rules.

---

# 6. Filter NULL Values

Instead of replacing them, keep only valid records.

### PySpark

```python
df = df.filter(df.salary.isNotNull())
```

### SQL

```sql
SELECT *
FROM employee
WHERE salary IS NOT NULL;
```

### Real-Time Example

Generate payroll only for employees with salary.

---

# 7. Detect NULL Values

Useful during data profiling.

### Spark

```python
from pyspark.sql.functions import col

df.filter(col("salary").isNull())
```

### SQL

```sql
SELECT *
FROM employee
WHERE salary IS NULL;
```

### Real-Time Example

Data quality validation.

---

# 8. Replace NULL Using Mean, Median or Average

Used in Machine Learning.

### PySpark

```python
avg_salary = df.selectExpr("avg(salary)").first()[0]

df = df.na.fill(avg_salary,["salary"])
```

### Real-Time Example

Customer income prediction.

---

# 9. Forward Fill (Last Non-NULL Value)

Useful for time-series data.

### PySpark

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import last

window = Window.orderBy("date") \
               .rowsBetween(Window.unboundedPreceding,0)

df = df.withColumn(
"value",
last("value",True).over(window)
)
```

### Real-Time Example

Stock market prices

IoT sensor readings

---

# 10. Backward Fill

Use the next available value.

### Spark

```python
window = Window.orderBy("date") \
.rowsBetween(0,Window.unboundedFollowing)

df = df.withColumn(
"value",
first("value",True).over(window)
)
```

### Real-Time Example

Forecasting datasets.

---

# 11. Replace NULL During Join

Sometimes joins create NULL values.

### SQL

```sql
SELECT

a.id,

COALESCE(b.salary,0)

FROM employee a

LEFT JOIN salary b

ON a.id=b.id;
```

### Real-Time Example

Dimension table missing records.

---

# 12. Handle NULL in Aggregations

Aggregate functions ignore NULL values.

### SQL

```sql
SELECT

AVG(salary)

FROM employee;
```

Average ignores NULLs.

To include NULL as zero

```sql
SELECT

AVG(COALESCE(salary,0))

FROM employee;
```

### Real-Time Example

Sales reports.

---

# 13. Handle NULL While Reading Data

Specify NULL representations.

### Spark

```python
df = (
spark.read
.option("nullValue","NULL")
.csv("/input")
)
```

### Real-Time Example

CSV files containing "NULL", "NA", or empty strings.

---

# 14. Handle NULL in Delta Lake MERGE

```sql
MERGE INTO target t

USING source s

ON t.id=s.id

WHEN MATCHED THEN

UPDATE SET

salary=COALESCE(s.salary,t.salary)
```

### Real-Time Example

Incremental data loads.

---

# 15. Replace Empty Strings with NULL

Empty string and NULL are different.

### PySpark

```python
from pyspark.sql.functions import when,col

df = df.withColumn(
"city",
when(col("city")=="",None)
.otherwise(col("city"))
)
```

### SQL

```sql
NULLIF(city,'')
```

### Real-Time Example

Legacy systems storing blanks instead of NULL.

---

# 16. Count NULL Values

Data quality check.

### Spark

```python
from pyspark.sql.functions import count,when,col

df.select(
count(
when(col("salary").isNull(),1)
).alias("Null_Count")
)
```

### SQL

```sql
SELECT

COUNT(*)

-

COUNT(salary)

FROM employee;
```

---

# 17. Drop Columns Having All NULL Values

### Spark

```python
df = df.na.drop(how="all")
```

### Real-Time Example

Raw ingestion files with unused columns.

---

# Real-Time Project Scenarios

| Scenario | Preferred Solution |
|----------|--------------------|
| Missing Customer ID | Drop record |
| Missing Salary | Replace with 0 |
| Missing City | Replace with "Unknown" |
| Missing Mobile | Use COALESCE(phone,mobile) |
| Sensor Data | Forward Fill |
| ML Features | Mean / Median Imputation |
| Blank Strings | Convert to NULL |
| LEFT JOIN NULLs | COALESCE() |
| CSV contains "NULL" | `option("nullValue","NULL")` |
| Data Quality Check | Count NULLs |

---

# Interview Questions

### Q1. Difference between NULL and Empty String?

| NULL | Empty String |
|------|--------------|
| Unknown or missing value | String with zero characters |
| `IS NULL` | `=''` |
| Consumes no value | Valid string value |

---

### Q2. Difference between `COALESCE`, `NVL`, and `IFNULL`?

| Function | Database |
|----------|----------|
| `COALESCE()` | ANSI SQL, Spark SQL, Hive, PostgreSQL, SQL Server |
| `NVL()` | Oracle, Hive |
| `IFNULL()` | MySQL |

- `COALESCE()` can evaluate multiple expressions.
- `NVL()` and `IFNULL()` typically compare two expressions.

---

### Q3. Does `AVG()` include NULL values?

No. Aggregate functions such as `AVG()`, `SUM()`, `MIN()`, and `MAX()` ignore NULL values. If you want NULLs treated as zero, wrap the column with `COALESCE()` or a similar function.

---

### Q4. How do you handle NULLs in production ETL?

- Identify mandatory columns and reject records with missing keys.
- Replace optional NULL values using business-defined defaults.
- Use `COALESCE()` during joins and transformations.
- Track NULL counts as part of data quality monitoring.
- Apply statistical imputation (mean/median) only for analytical or ML workloads.

---

# Best Practices

- ✅ Define mandatory columns (for example, primary keys) and reject records where they are NULL.
- ✅ Use `COALESCE()` as the preferred ANSI SQL approach for replacing NULL values.
- ✅ Treat empty strings (`""`) and NULL values separately, converting blanks to NULL where appropriate.
- ✅ Validate and report NULL counts during every ETL pipeline execution.
- ✅ Use forward fill or backward fill only for ordered, time-series data where it makes business sense.
- ✅ Avoid replacing NULLs with arbitrary defaults unless those defaults are defined by business rules.
- ✅ Handle NULL values as early as possible in the data pipeline to improve downstream data quality.